## 3. Data Loading & Graph Construction

In this section we:
1. Parse the two real GPX files into Python data structures
2. Explore and visualize the raw data (elevation profiles)
3. Convert the GPS points into a **weighted graph**

---

### 3.1 What is a GPX file?

A **GPX (GPS Exchange Format)** file is an XML-based format for storing GPS data.
Each track point (`<trkpt>`) contains:

```xml
<trkpt lat="42.1234" lon="23.5678">
    <ele>2100.5</ele>
</trkpt>
```

- `lat` — latitude in decimal degrees
- `lon` — longitude in decimal degrees  
- `ele` — elevation above sea level in meters

We will parse these using Python's built-in `xml.etree.ElementTree` — no external libraries needed.

---

In [ ]:
# ============================================================
# Section 3.1 — GPX Parser
# ============================================================

import xml.etree.ElementTree as ET
import math
import numpy as np
import matplotlib.pyplot as plt


def parse_gpx(filepath):
    """
    Parse a GPX file and return a list of (lat, lon, elevation) tuples.

    Parameters:
        filepath : str — path to .gpx file

    Returns:
        List of tuples: [(lat, lon, ele), ...]
    """
    tree = ET.parse(filepath)
    root = tree.getroot()

    # GPX files use a namespace — we must include it in all searches
    ns = {'gpx': 'http://www.topografix.com/GPX/1/1'}

    points = []
    for trkpt in root.findall('.//gpx:trkpt', ns):
        lat = float(trkpt.get('lat'))
        lon = float(trkpt.get('lon'))
        ele_tag = trkpt.find('gpx:ele', ns)
        ele = float(ele_tag.text) if ele_tag is not None else 0.0
        points.append((lat, lon, ele))

    return points


# Load both tracks
RILA_PATH   = '../data/bulgaria-rila-mountains-2-day-hike-rila-monastery-seven-lake.gpx'
VIHREN_PATH = '../data/Vihren-Pirin'

rila_points   = parse_gpx(RILA_PATH)
vihren_points = parse_gpx(VIHREN_PATH)

print(f'Rila   — loaded {len(rila_points):,} GPS points')
print(f'Vihren — loaded {len(vihren_points):,} GPS points')
print()
print('First point (Rila):  ', rila_points[0])
print('Last  point (Rila):  ', rila_points[-1])
print('First point (Vihren):', vihren_points[0])
print('Last  point (Vihren):', vihren_points[-1])

In [ ]:
# ============================================================
# Section 3.2 — Track Statistics
# ============================================================

def haversine(lat1, lon1, lat2, lon2):
    """Great-circle distance between two GPS points (meters)."""
    R = 6_371_000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi    = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = (math.sin(dphi/2)**2
         + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda/2)**2)
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def track_statistics(points, name):
    """
    Compute key statistics for a GPS track.

    Returns dict with distance, elevation gain/loss, min/max elevation.
    """
    total_dist  = 0.0
    elev_gain   = 0.0
    elev_loss   = 0.0

    for i in range(1, len(points)):
        lat1, lon1, e1 = points[i-1]
        lat2, lon2, e2 = points[i]
        total_dist += haversine(lat1, lon1, lat2, lon2)
        diff = e2 - e1
        if diff > 0:
            elev_gain += diff
        else:
            elev_loss += abs(diff)

    elevations   = [p[2] for p in points]
    naismith_hrs = (total_dist/1000)/5 + elev_gain/600

    stats = {
        'name'          : name,
        'num_points'    : len(points),
        'distance_km'   : round(total_dist / 1000, 2),
        'elevation_gain': round(elev_gain, 0),
        'elevation_loss': round(elev_loss, 0),
        'max_elevation' : round(max(elevations), 0),
        'min_elevation' : round(min(elevations), 0),
        'naismith_hours': round(naismith_hrs, 2),
    }

    print(f'Track: {name}')
    print(f'  GPS points    : {stats["num_points"]:,}')
    print(f'  Distance      : {stats["distance_km"]} km')
    print(f'  Elevation gain: +{stats["elevation_gain"]:.0f} m')
    print(f'  Elevation loss: -{stats["elevation_loss"]:.0f} m')
    print(f'  Max elevation : {stats["max_elevation"]:.0f} m')
    print(f'  Min elevation : {stats["min_elevation"]:.0f} m')
    print(f'  Naismith time : {stats["naismith_hours"]} hours')
    print()
    return stats


rila_stats   = track_statistics(rila_points,   'Rila Monastery -> Seven Lakes')
vihren_stats = track_statistics(vihren_points, 'Vihren Peak (Pirin)')

In [ ]:
# ============================================================
# Section 3.3 — Elevation Profile Visualization
# ============================================================

def cumulative_distances(points):
    """Return list of cumulative distances (km) for each point."""
    dists = [0.0]
    for i in range(1, len(points)):
        lat1, lon1, _ = points[i-1]
        lat2, lon2, _ = points[i]
        dists.append(dists[-1] + haversine(lat1, lon1, lat2, lon2) / 1000)
    return dists


fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('Elevation Profiles — Real Bulgarian Mountain Tracks',
             fontsize=14, fontweight='bold')

tracks_plot = [
    (rila_points,   'Rila: Monastery → Seven Lakes', '#2ECC71', axes[0]),
    (vihren_points, 'Vihren Peak — Pirin',           '#E74C3C', axes[1]),
]

for points, title, color, ax in tracks_plot:
    distances  = cumulative_distances(points)
    elevations = [p[2] for p in points]

    ax.fill_between(distances, elevations, min(elevations),
                    alpha=0.25, color=color)
    ax.plot(distances, elevations, color=color, linewidth=1.2)

    # Mark start and end
    ax.scatter([distances[0]],  [elevations[0]],
               color='green', zorder=5, s=80, label='Start')
    ax.scatter([distances[-1]], [elevations[-1]],
               color='red',   zorder=5, s=80, label='End')

    # Mark highest point
    max_idx = elevations.index(max(elevations))
    ax.scatter([distances[max_idx]], [elevations[max_idx]],
               color='gold', zorder=5, s=100,
               label=f'Peak: {max(elevations):.0f}m')

    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Distance (km)')
    ax.set_ylabel('Elevation (m)')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('Elevation profiles plotted!')

### 3.4 Building the Graph

We now convert each GPS track into a **weighted directed graph**.

Each GPS point becomes a **node** with index $i$.
Each consecutive pair of points $(i, i+1)$ becomes an **edge** with three possible weights:

| Weight | Formula |
|---|---|
| Distance | `haversine(p_i, p_{i+1})` in meters |
| Elevation gain | `max(0, ele_{i+1} - ele_i)` in meters |
| Naismith time | `dist/5000 + max(0, Δele)/600` in hours |

The graph is stored as an **adjacency list** — a dictionary where each key is a node,
and the value is a list of `(neighbor, weights)` tuples.

---

In [ ]:
# ============================================================
# Section 3.4 — Graph Construction
# ============================================================

def build_graph(points):
    """
    Build a weighted directed graph from a list of GPS points.

    Each node is an integer index (0 to N-1).
    Each edge (i -> i+1) has three weights:
      - 'dist'    : Haversine distance in meters
      - 'elev'    : Elevation gain (max 0, delta_ele) in meters
      - 'naismith': Estimated hiking time in hours

    Returns:
        graph : dict { node_id: [(neighbor_id, dist, elev, naismith), ...] }
        nodes : list of (lat, lon, ele) — same order as indices
    """
    graph = {i: [] for i in range(len(points))}

    for i in range(len(points) - 1):
        lat1, lon1, e1 = points[i]
        lat2, lon2, e2 = points[i + 1]

        dist     = haversine(lat1, lon1, lat2, lon2)
        elev     = max(0.0, e2 - e1)
        naismith = (dist / 1000) / 5 + elev / 600

        # Forward edge: i -> i+1
        graph[i].append((i + 1, dist, elev, naismith))

        # Backward edge: i+1 -> i (going back is also possible)
        elev_back     = max(0.0, e1 - e2)
        naismith_back = (dist / 1000) / 5 + elev_back / 600
        graph[i + 1].append((i, dist, elev_back, naismith_back))

    return graph, points


# Build graphs for both tracks
rila_graph,   rila_nodes   = build_graph(rila_points)
vihren_graph, vihren_nodes = build_graph(vihren_points)

# Inspect a sample edge
print('Graph construction complete!')
print()
print(f'Rila graph   : {len(rila_graph)} nodes, {sum(len(v) for v in rila_graph.values())} edges')
print(f'Vihren graph : {len(vihren_graph)} nodes, {sum(len(v) for v in vihren_graph.values())} edges')
print()
print('Sample edge (Rila, node 0 -> node 1):')
neighbor, dist, elev, naismith = rila_graph[0][0]
print(f'  To node     : {neighbor}')
print(f'  Distance    : {dist:.2f} m')
print(f'  Elev gain   : {elev:.2f} m')
print(f'  Naismith    : {naismith*60:.3f} min')